# 复现 Fig. 18：PBH 质量函数

Fig. 18 展示论文 Appendix B 使用的 PBH 质量分布。通用概率密度公式保存在 `pbhmassfunction.py`，本 Notebook 只负责选取论文参数、绘制两个面板并导出数据。

- 上面板：log-normal、broken power-law、critical collapse；
- 下面板：固定中位质量 $30 M_\odot$，比较 log-normal 的 $\sigma=0.4,0.6,0.8$；
- 所有分布都按照图注在 $5$–$150 M_\odot$ 上归一化。

## 1. 导入模块并定位项目目录

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pbhmassfunction.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pbhmassfunction as pmf

PROJECT_ROOT

## 2. 设置质量网格与论文参数

横轴使用线性质量网格。1000 个点足以显示 broken power-law 在 $m_*=31.1 M_\odot$ 处的尖锐转折。

critical-collapse 分布存在一处文图冲突：Eq. (B4) 后的正文写 $\alpha=1$，Fig. 18 图注写 $\alpha=1.1$。这里复现具体图片，因此采用图注值 $1.1$。

In [ ]:
MASS_MIN_MSUN = 5.0
MASS_MAX_MSUN = 150.0
MASS_GRID_MSUN = np.linspace(MASS_MIN_MSUN, MASS_MAX_MSUN, 1000)

LOGNORMAL_MEDIAN_MSUN = 30.0
TOP_LOGNORMAL_SIGMA = 0.71
BROKEN_PEAK_MASS_MSUN = 31.1
BROKEN_ALPHA1 = 0.54
BROKEN_ALPHA2 = 5.6
CRITICAL_MASS_SCALE_MSUN = 10.8
CRITICAL_ALPHA = 1.1
BOTTOM_LOGNORMAL_SIGMAS = (0.4, 0.6, 0.8)

MASS_GRID_MSUN[[0, 250, 500, 750, 999]]

## 3. 计算上面板的三种质量分布

这些函数返回的是关于线性质量测度 $dm$ 的概率密度 $\psi(m)$，单位为 $M_\odot^{-1}$。因此概率质量是 $\psi(m)dm$，不要把它和关于 $d\ln m$ 定义的密度混用。

In [ ]:
top_lognormal = pmf.lognormal_mass_pdf(
    MASS_GRID_MSUN,
    median_mass_msun=LOGNORMAL_MEDIAN_MSUN,
    sigma=TOP_LOGNORMAL_SIGMA,
)
broken_power_law = pmf.broken_power_law_mass_pdf(
    MASS_GRID_MSUN,
    peak_mass_msun=BROKEN_PEAK_MASS_MSUN,
    alpha1=BROKEN_ALPHA1,
    alpha2=BROKEN_ALPHA2,
)
critical_collapse = pmf.critical_collapse_mass_pdf(
    MASS_GRID_MSUN,
    mass_scale_msun=CRITICAL_MASS_SCALE_MSUN,
    alpha=CRITICAL_ALPHA,
)

# 三列分别对应 Fig. 18 上面板的三条曲线；这里只显示五个质量点。
sample_indices = [0, 250, 500, 750, 999]
np.column_stack([
    MASS_GRID_MSUN[sample_indices],
    top_lognormal[sample_indices],
    broken_power_law[sample_indices],
    critical_collapse[sample_indices],
])

## 4. 计算下面板的三种 log-normal 宽度

三条曲线具有相同中位质量。$\sigma$ 越大，分布在 $\ln m$ 空间越宽，因此在线性质量横轴上峰值降低、高质量尾部变长。

In [ ]:
bottom_lognormal = {
    sigma: pmf.lognormal_mass_pdf(
        MASS_GRID_MSUN,
        median_mass_msun=LOGNORMAL_MEDIAN_MSUN,
        sigma=sigma,
    )
    for sigma in BOTTOM_LOGNORMAL_SIGMAS
}

np.column_stack([
    MASS_GRID_MSUN[sample_indices],
    bottom_lognormal[0.4][sample_indices],
    bottom_lognormal[0.6][sample_indices],
    bottom_lognormal[0.8][sample_indices],
])

## 5. 绘制 Fig. 18 的两个面板

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(7.0, 7.4))

axes[0].plot(
    MASS_GRID_MSUN,
    top_lognormal,
    color="orange",
    linestyle="-",
    label="log-normal",
)
axes[0].plot(
    MASS_GRID_MSUN,
    broken_power_law,
    color="tab:green",
    linestyle="-.",
    label="broken power-law",
)
axes[0].plot(
    MASS_GRID_MSUN,
    critical_collapse,
    color="tab:red",
    linestyle=":",
    label="critical collapse",
)

bottom_styles = {
    0.4: ("orange", "-"),
    0.6: ("tab:green", "-."),
    0.8: ("tab:red", ":"),
}
for sigma in BOTTOM_LOGNORMAL_SIGMAS:
    color, line_style = bottom_styles[sigma]
    axes[1].plot(
        MASS_GRID_MSUN,
        bottom_lognormal[sigma],
        color=color,
        linestyle=line_style,
        label=rf"$\sigma={sigma}$",
    )

axes[0].set_ylim(-0.002, 0.045)
axes[1].set_ylim(-0.001, 0.0375)
axes[0].set_yticks([0.00, 0.01, 0.02, 0.03, 0.04])
axes[1].set_yticks([0.00, 0.01, 0.02, 0.03])

for axis in axes:
    axis.set_xlim(MASS_MIN_MSUN, MASS_MAX_MSUN)
    axis.set_xticks(np.arange(25.0, 151.0, 25.0))
    axis.set_xlabel(r"PBH Mass ($M_\odot$)")
    axis.set_ylabel(r"$\psi(m)$")
    axis.legend(loc="upper right")

figure.tight_layout(h_pad=2.0)
figure_path = PROJECT_ROOT / "fig18_reproduction.png"
figure.savefig(figure_path, dpi=240, bbox_inches="tight")
print(f"图片已保存：{figure_path}")
plt.show()

## 6. 导出绘图数据

CSV 保存统一质量网格和六条曲线，后续改变图片样式时可以直接读取，不必重新计算。

In [ ]:
figure18_table = np.column_stack([
    MASS_GRID_MSUN,
    top_lognormal,
    broken_power_law,
    critical_collapse,
    bottom_lognormal[0.4],
    bottom_lognormal[0.6],
    bottom_lognormal[0.8],
])

csv_path = PROJECT_ROOT / "fig18_reproduction.csv"
np.savetxt(
    csv_path,
    figure18_table,
    delimiter=",",
    header=(
        "mass_msun,"
        "lognormal_sigma_0p71,"
        "broken_power_law,"
        "critical_collapse_alpha_1p1,"
        "lognormal_sigma_0p4,"
        "lognormal_sigma_0p6,"
        "lognormal_sigma_0p8"
    ),
    comments="",
    fmt="%.10e",
)

print(f"数据已保存：{csv_path}")